# Practica B4-T2 - XAI aplicado a concesion de credito

**Notebook UNICO de entrega**, consolidado a partir de los notebooks de trabajo `notebooks/01_EDA.ipynb` a `notebooks/07_shap.ipynb` del repositorio `TAREA_XAI`.

Este documento es el unico artefacto que se entrega en el aula virtual junto con los dos ficheros de predicciones de produccion (`results/predicciones/cs_produccion1.csv` y `results/predicciones/cs_produccion2.csv`), tal y como exige el enunciado de la practica (`docs/_fuentes/taller_XAI.pdf` / `taller_XAI.pdf`).

**Indice de secciones:**
1. Portada
2. Objetivo y planteamiento del problema
3. Datos y EDA (consolida `01_EDA.ipynb`)
4. Preprocesado (consolida `02_preprocesado.ipynb`)
5. Modelo, escenario de coste FP = FN = 1 (consolida `03_modelo_coste1.ipynb`)
6. Modelo, escenario de coste FP = FN = 10 (consolida `04_modelo_coste10.ipynb`)
7. Tabla comparativa de resultados entre ambos escenarios
8. Auditoria: modelo subrogado con reglas (consolida `05_auditoria_subrogado.ipynb`)
9. Auditoria: analisis de contrafactuales (consolida `06_contrafactuals.ipynb`)
10. Auditoria: analisis SHAP global y local (consolida `07_shap.ipynb`)
11. Otras tecnicas de explicabilidad consideradas (opcional)
12. Decisiones de diseno (`docs/DECISIONES.md`)
13. Reflexion final

## 1. Portada

**Master MIAX - Bloque 4 - Practica B4-T2. XAI**

**Componentes del grupo:**
- Nombre Apellido1 Apellido2 (email@dominio.com)
- Nombre Apellido1 Apellido2 (email@dominio.com)
- Nombre Apellido1 Apellido2 (email@dominio.com)

*(placeholders — sustituir por los nombres y emails reales de los integrantes del grupo antes de la entrega).*

**Fecha de entrega:** 20 de julio de 2026 (antes de las 23:59, a traves del aula virtual).

**Entregables de esta practica:**
- Este notebook: `99_ENTREGA.ipynb`.
- `results/predicciones/cs_produccion1.csv` — predicciones de produccion, escenario FP = FN = 1.
- `results/predicciones/cs_produccion2.csv` — predicciones de produccion, escenario FP = FN = 10.

## 2. Objetivo y planteamiento del problema

El objetivo de la practica (ver `taller_XAI.pdf`, seccion 1) es **construir, auditar y optimizar un modelo de concesion de credito** bajo dos condiciones de coste distintas:

- **Escenario de coste 1:** Coste Falso Positivo = Coste Falso Negativo = 1. Las predicciones sobre el dataset de produccion se entregan en `cs_produccion1.csv`.
- **Escenario de coste 2:** Coste Falso Positivo = Coste Falso Negativo = 10. Las predicciones sobre el dataset de produccion se entregan en `cs_produccion2.csv`.

Ademas de construir y optimizar el modelo en ambos escenarios, el enunciado exige auditar el/los modelo(s) resultante(s) mediante:

- Un **modelo subrogado con reglas** que explique el comportamiento del modelo caja negra (seccion 8).
- Un **analisis de contrafactuales** para varios ejemplos de clase real 0 y 1, respondiendo a la pregunta: si un cliente pide explicaciones sobre por que se le deniega un credito, ¿que informacion le damos? (seccion 9).
- Un **analisis SHAP global y local** (seccion 10).
- **Otras tecnicas** de explicabilidad que el grupo considere oportunas (seccion 11, opcional).

Los modelos candidatos admitidos por el enunciado son: a) un modelo supervisado (clasificacion binaria), o b) un Multiarmed Bandit. La decision sobre cual de los dos se adopta finalmente se documenta en `docs/DECISIONES.md` (**D-0.1**, ver seccion 12).

**Criterio de evaluacion** (`taller_XAI.pdf`, seccion 6), 50 % + 50 %:
- **Resultados (50 %):** coste promedio del modelo en el dataset de produccion.
- **Analisis, auditoria y explicaciones (50 %):** calidad del analisis presentado, coherencia y capacidad de reflexion critica.

Este notebook consolida, en un unico documento, el trabajo desarrollado de forma incremental en los notebooks `notebooks/01_EDA.ipynb` a `notebooks/07_shap.ipynb`, siguiendo el contrato de rutas y artefactos compartido entre todos ellos (datos en `data/`, artefactos intermedios en `data/processed/` y `results/`, ver seccion de Conectores de cada notebook de trabajo).

In [ ]:
# TODO: definir como constantes de referencia, para su uso a lo largo de todo este notebook,
# las dos matrices de coste del enunciado (FP=FN=1 y FP=FN=10), apoyandose en las utilidades
# de src/cost_utils.py (ver tambien docs/teoria/cost_sensitive.md).

## 3. Datos y EDA

**Esta seccion consolida el trabajo de `notebooks/01_EDA.ipynb`.**

Resumen del analisis exploratorio sobre `data/cs_construccion.csv` (train, ~105.000 filas, target `SeriousDlqin2yrs`), `data/cs_produccion.csv` (produccion/scoring, ~45.000 filas, sin target) y `data/DataDictionary.csv` (diccionario de columnas): balance de clases del target, nulos y valores atipicos por variable (en particular `MonthlyIncome`, `NumberOfDependents`, `DebtRatio`, `RevolvingUtilizationOfUnsecuredLines` y `age`), distribucion de variables y su relacion con el target, y matriz de correlaciones. Conclusiones y recomendaciones de cara al preprocesado, consolidadas en `results/tables/eda_resumen.csv`.

In [ ]:
# TODO: consolidar aqui el codigo y las figuras clave de notebooks/01_EDA.ipynb:
#   - carga de data/cs_construccion.csv, data/cs_produccion.csv, data/DataDictionary.csv
#   - analisis de SeriousDlqin2yrs y desbalance de clases
#   - nulos y outliers por variable
#   - distribuciones y relacion bivariada con el target
#   - matriz de correlaciones
#   - carga/lectura de results/tables/eda_resumen.csv con las conclusiones consolidadas

## 4. Preprocesado

**Esta seccion consolida el trabajo de `notebooks/02_preprocesado.ipynb`.**

Construccion del pipeline de preprocesado (division train/test estratificada, imputacion de nulos, tratamiento de outliers, encoding/escalado de variables) implementado en `src/preprocessing.py`, ajustado (`fit`) una unica vez sobre train y serializado en `results/models/preprocessing_pipeline.joblib`. Verificacion de que `transform` es identico entre train, test y produccion, generando `data/processed/train.parquet`, `data/processed/test.parquet` y `data/processed/produccion.parquet`, artefactos consumidos por los modelos de las secciones 5 y 6 y por las auditorias de las secciones 8-10.

In [ ]:
# TODO: consolidar aqui el codigo de notebooks/02_preprocesado.ipynb:
#   - division train/test estratificada de data/cs_construccion.csv
#   - imputacion de nulos (MonthlyIncome, NumberOfDependents)
#   - tratamiento de outliers (DebtRatio, RevolvingUtilizationOfUnsecuredLines, age)
#   - encoding/escalado segun la familia de modelo elegida (D-0.3)
#   - fit y serializacion del pipeline en results/models/preprocessing_pipeline.joblib
#   - verificacion de transform identico y guardado de los parquet en data/processed/

## 5. Modelo — escenario de coste FP = FN = 1

**Esta seccion consolida el trabajo de `notebooks/03_modelo_coste1.ipynb`.**

Entrenamiento del modelo de concesion de credito (familia definida por **D-0.3**) sobre `data/processed/train.parquet`, calculo del coste esperado con la matriz de coste FP = FN = 1 (`src/cost_utils.py`), optimizacion del umbral de decision que minimiza dicho coste, evaluacion sobre `data/processed/test.parquet` (coste promedio, matriz de confusion, metricas de clasificacion) y generacion de las predicciones de produccion `results/predicciones/cs_produccion1.csv`. El modelo se serializa en `results/models/modelo_coste1.joblib`.

**Tabla de resultados (escenario FP = FN = 1):** umbral optimo, coste esperado en test y metricas de clasificacion complementarias, tal y como se registran en la fila `coste1` de `results/tables/umbrales_coste.csv`.

In [ ]:
# TODO: consolidar aqui el codigo de notebooks/03_modelo_coste1.ipynb:
#   - carga de data/processed/{train,test,produccion}.parquet y del pipeline
#   - entrenamiento del modelo (familia segun D-0.3) y predict_proba
#   - calculo del coste esperado (FP=FN=1) con src/cost_utils.py
#   - optimizacion del umbral de decision (ver propuesta D-3.1)
#   - evaluacion en test: coste promedio, matriz de confusion, metricas
#   - tabla de resultados del escenario (fila 'coste1' de results/tables/umbrales_coste.csv)
#   - generacion de results/predicciones/cs_produccion1.csv
#   - serializacion del modelo en results/models/modelo_coste1.joblib

## 6. Modelo — escenario de coste FP = FN = 10

**Esta seccion consolida el trabajo de `notebooks/04_modelo_coste10.ipynb`.**

Analogo a la seccion 5, pero bajo la matriz de coste FP = FN = 10: reutilizacion o reentrenamiento del modelo segun como se resuelva **D-0.2** (un unico modelo de scoring con dos umbrales vs. dos modelos independientes), calculo del coste esperado con la matriz FP = FN = 10, busqueda del umbral optimo para este escenario, evaluacion sobre `data/processed/test.parquet` y generacion de las predicciones de produccion `results/predicciones/cs_produccion2.csv`. El modelo (o los parametros del umbral, segun D-0.2) se serializa en `results/models/modelo_coste10.joblib`.

**Tabla de resultados (escenario FP = FN = 10):** umbral optimo, coste esperado en test y metricas de clasificacion complementarias, tal y como se registran en la fila `coste10` de `results/tables/umbrales_coste.csv`.

In [ ]:
# TODO: consolidar aqui el codigo de notebooks/04_modelo_coste10.ipynb:
#   - carga de data/processed/{train,test,produccion}.parquet y del pipeline
#   - reutilizacion o reentrenamiento del modelo segun D-0.2
#   - calculo del coste esperado (FP=FN=10) con src/cost_utils.py
#   - optimizacion del umbral de decision para este escenario
#   - evaluacion en test: coste promedio, matriz de confusion, metricas
#   - tabla de resultados del escenario (fila 'coste10' de results/tables/umbrales_coste.csv)
#   - generacion de results/predicciones/cs_produccion2.csv
#   - serializacion del modelo/umbral en results/models/modelo_coste10.joblib

## 7. Tabla comparativa de resultados entre ambos escenarios

Tabla comparativa que combina las filas `coste1` y `coste10` de `results/tables/umbrales_coste.csv` (secciones 5 y 6): umbral optimo, coste esperado en test y **coste promedio en el dataset de produccion** para cada escenario (a partir de `results/predicciones/cs_produccion1.csv` y `results/predicciones/cs_produccion2.csv`).

Esta tabla es el soporte directo del primer criterio de evaluacion del enunciado (*Resultados, 50 % de la nota: coste promedio del modelo en el dataset de produccion*, `taller_XAI.pdf` seccion 6), por lo que debe quedar claramente visible el coste promedio de produccion de cada escenario, y una breve discusion de por que difieren (o no) el umbral optimo y el comportamiento del modelo entre ambas matrices de coste.

In [ ]:
# TODO: construir la tabla comparativa final (coste1 vs coste10):
#   - leer results/tables/umbrales_coste.csv (ambas filas)
#   - calcular el coste promedio en produccion de cada escenario a partir de
#     results/predicciones/cs_produccion1.csv y results/predicciones/cs_produccion2.csv
#   - presentar la tabla comparativa (umbral, coste esperado test, coste promedio produccion)
#   - breve discusion cuantitativa de las diferencias entre ambos escenarios de coste

## 8. Auditoria: modelo subrogado con reglas

**Esta seccion consolida el trabajo de `notebooks/05_auditoria_subrogado.ipynb`.**

Ajuste de un modelo subrogado interpretable (arbol de decision) que aproxime las predicciones de cada uno de los dos modelos de caja negra (`modelo_coste1` y `modelo_coste10`), extraccion de reglas legibles (if/then, con soporte y pureza por hoja) guardadas en `results/tables/reglas_subrogado_coste1.md` y `results/tables/reglas_subrogado_coste10.md`, medicion de la fidelidad del subrogado frente al modelo original (no frente al target real), e interpretacion cualitativa de las reglas obtenidas en cada escenario de coste.

In [ ]:
# TODO: consolidar aqui el codigo de notebooks/05_auditoria_subrogado.ipynb:
#   - carga de results/models/modelo_coste1.joblib, modelo_coste10.joblib y data/processed/test.parquet
#   - ajuste del arbol subrogado para el escenario coste1
#   - ajuste del arbol subrogado para el escenario coste10
#   - extraccion de reglas legibles (reglas_subrogado_coste{1,10}.md)
#   - medicion de fidelidad del subrogado frente al modelo caja negra
#   - interpretacion de las reglas obtenidas por escenario

## 9. Auditoria: analisis de contrafactuales

**Esta seccion consolida el trabajo de `notebooks/06_contrafactuals.ipynb`.**

Seleccion de varios ejemplos de clase real 0 y clase real 1 (en ambos escenarios de coste), generacion de contrafactuales (instancias hipoteticas proximas con prediccion opuesta) y validacion de su plausibilidad frente al rango de valores reales del dataset. Traduccion de los contrafactuales validados a una **explicacion en lenguaje no tecnico**, respondiendo directamente a la pregunta del enunciado: si un cliente pide explicaciones sobre por que se le deniega un credito, ¿que informacion le damos? Ejemplos guardados en `results/tables/contrafactuals_ejemplos.csv`.

In [ ]:
# TODO: consolidar aqui el codigo de notebooks/06_contrafactuals.ipynb:
#   - seleccion de ejemplos de clase real 0 y 1 (ambos escenarios de coste)
#   - generacion de contrafactuales para cada ejemplo seleccionado
#   - validacion de plausibilidad de los contrafactuales generados
#   - traduccion de los contrafactuales a una explicacion apta para el cliente
#   - guardado de resultados en results/tables/contrafactuals_ejemplos.csv

## 10. Auditoria: analisis SHAP global y local

**Esta seccion consolida el trabajo de `notebooks/07_shap.ipynb`.**

Calculo de valores SHAP (Explainer segun la familia de modelo de **D-0.3**) para los dos modelos de caja negra, visualizacion de la importancia global de variables (summary plots, `results/figures/`) y calculo de valores SHAP locales para los mismos ejemplos individuales analizados en la seccion 9 (contrafactuales), permitiendo contrastar ambas explicaciones. Comparacion de los hallazgos SHAP entre el escenario FP = FN = 1 y el escenario FP = FN = 10. Valores globales consolidados en `results/tables/shap_values_global.csv`.

In [ ]:
# TODO: consolidar aqui el codigo de notebooks/07_shap.ipynb:
#   - carga de results/models/modelo_coste1.joblib, modelo_coste10.joblib y data/processed/test.parquet
#   - calculo de valores SHAP globales (ambos modelos)
#   - visualizacion de importancia global de variables (summary plots)
#   - calculo de valores SHAP locales para los ejemplos de la seccion 9
#   - comparacion de hallazgos SHAP entre escenario coste1 y coste10
#   - guardado de results/tables/shap_values_global.csv

## 11. Otras tecnicas de explicabilidad consideradas (opcional)

El enunciado (`taller_XAI.pdf`, seccion 1) invita explicitamente a incorporar *"otras tecnicas que los estudiantes consideren oportunas"* ademas del subrogado, los contrafactuales y SHAP. Esta seccion es **opcional** y recoge, si el grupo decide desarrollarlas, tecnicas complementarias de explicabilidad tales como: graficos de dependencia parcial (PDP) / ICE, importancia por permutacion, explicaciones locales tipo LIME o anchors, o un analisis de equidad/sesgo del modelo respecto a variables sensibles (p. ej. `age`). Si finalmente no se desarrolla ninguna tecnica adicional, dejar constancia explicita de esa decision y su justificacion.

In [ ]:
# TODO: (opcional) implementar y documentar aqui otras tecnicas de explicabilidad adicionales
# consideradas por el grupo (p. ej. PDP/ICE, importancia por permutacion, LIME, anchors,
# analisis de equidad/sesgo); o razonar explicitamente por que no se ha considerado necesario
# anadir ninguna tecnica mas alla de subrogado + contrafactuales + SHAP.

## 12. Decisiones de diseno (`docs/DECISIONES.md`)

Resumen final de las decisiones transversales que condicionan todo el pipeline, registradas con formato tipo ADR en `docs/DECISIONES.md`. A fecha de creacion de este esqueleto las tres estan **ABIERTA**; antes de la entrega deben quedar **CERRADA**s con su justificacion final (evidencia empirica obtenida en las secciones 3-10):

- **D-0.1** — ¿Modelo supervisado o Multiarmed Bandit? Alternativas: clasificacion binaria clasica sobre datos historicos etiquetados vs. aprendizaje online con exploracion/explotacion. Resolucion final y justificacion: *(pendiente de completar)*.
- **D-0.2** — ¿Un unico modelo de scoring con dos umbrales de decision (uno por escenario de coste) o dos modelos entrenados de forma independiente? Resolucion final y justificacion: *(pendiente de completar)*.
- **D-0.3** — Familia de modelo (arboles boosted vs. red neuronal vs. modelo lineal), en funcion del trade-off rendimiento/explicabilidad observado. Resolucion final y justificacion: *(pendiente de completar)*.

Ademas, cada notebook de trabajo (`02` a `07`) puede haber propuesto decisiones especificas propias con prefijo `D-<numero de notebook>.n` (p. ej. `D-2.1`, `D-3.1`, ...) en su propia seccion de *Decisiones que afectan a este notebook*; resumir aqui unicamente las que resulten relevantes para entender los resultados finales.

In [ ]:
# TODO: cargar/leer docs/DECISIONES.md y resumir en una tabla el estado final (CERRADA/ABIERTA)
# de D-0.1, D-0.2, D-0.3 y de las decisiones especificas de notebook (D-2.x, D-3.x, ...) que
# resulten relevantes, junto con la justificacion final adoptada en cada caso.

## 13. Reflexion final

*(Seccion pendiente de completar una vez se disponga de resultados reales en las secciones 3-12.)*

Esta reflexion final debe cubrir, como minimo, lo exigido por el enunciado (*"una reflexion final sobre los resultados obtenidos"*, `taller_XAI.pdf` seccion 4) y debe dar cuenta, con capacidad de reflexion critica, de:

- Como cambia el comportamiento optimo del modelo (umbral, tasa de aprobacion, coste promedio en produccion) al pasar del escenario FP = FN = 1 al escenario FP = FN = 10, y por que.
- El trade-off observado entre rendimiento/coste y explicabilidad segun la familia de modelo finalmente elegida (**D-0.3**).
- Que aprendizajes deja la auditoria (subrogado, contrafactuales, SHAP y, en su caso, otras tecnicas de la seccion 11) sobre la fiabilidad y las limitaciones del modelo, y sobre la calidad/utilidad de las explicaciones que se podrian dar a un cliente al que se deniega un credito.
- Limitaciones del analisis realizado y posibles lineas de mejora futuras.